# Feature Engineering — Joint XGBoost + Block Optuna

Uses the **Optuna-tuned XGBoost** setup from `models copy.ipynb`:
- Hyperparameter search space: `build_xgb_classifier()`
- Saved baseline: `optuna_best_extended.json`
- Base feature blocks: `FEATURE_BLOCKS` from models copy

This notebook adds **new feature-engineering blocks** and runs:
1. Fixed-parameter ablation (leave-one-out + add-one-new-block)
2. Joint Optuna: **XGBoost params + all ablation block toggles**

Metric: 5-fold stratified CV log loss (primary), AUC ROC (secondary).

## 1. Imports

Helper modules (same folder as notebook):
- `model_copy_utils.py` — XGB tuning setup from models copy
- `feature_eng_lib.py` — ablation blocks + submission helpers

Run this cell first. Section 8 can run standalone after cells 1, 2, and 4 if `feature_eng_joint_best.json` exists.

In [23]:
import json
import sys
from pathlib import Path

import optuna
import pandas as pd

# Ensure project root is on path when running from notebook
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_copy_utils import (
    CV_FOLDS,
    FEATURE_BLOCKS,
    MODEL_BUILDERS,
    OPTUNA_BEST_PATH,
    OPTUNA_CV_FOLDS,
    OPTUNA_RANDOM_STATE,
    RANDOM_STATE,
    build_tuned_xgb,
    build_xgb_classifier,
    evaluate_xgb_cv,
    get_feature_sets,
    load_data,
    load_optuna_best,
    load_xgb_best,
)
from feature_eng_lib import (
    ALL_ABLATION_BLOCKS,
    BASE_ABLATION_BLOCKS,
    NEW_ABLATION_BLOCKS,
    OPTUNA_N_TRIALS,
    OPTUNA_STORAGE,
    OPTUNA_BEST_PATH as FEATURE_ENG_BEST_PATH,
    all_blocks_active,
    build_feature_cols_from_blocks,
    engineer_all_features,
    evaluate_block_config,
    load_feature_eng_best,
    make_feature_eng_submission,
    make_joint_optuna_objective,
    run_fixed_params_ablation,
    store_feature_eng_best,
    summarize_block_effects,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Load tuned XGBoost from models copy

Reference model: best XGB trial from `models copy.ipynb` (sections 10–12).

In [24]:
xgb_best = load_xgb_best()
XGB_PARAMS = xgb_best["params"]
MODELS_COPY_FEATURE_COLS = xgb_best["feature_cols"]

tuned_xgb = MODEL_BUILDERS["xgboost"](XGB_PARAMS)

train_df = engineer_all_features(load_data("train.csv"))
y = train_df["default"]

drop_cols = ["client_id", "default"]
all_feature_cols = [c for c in train_df.columns if c not in drop_cols]
FEATURE_SETS = get_feature_sets(all_feature_cols)

models_copy_baseline = evaluate_xgb_cv(
    tuned_xgb, train_df[MODELS_COPY_FEATURE_COLS], y
)

print(f"Models copy tuned feature label: {xgb_best['feature_label']}")
print(f"Models copy tuned val log loss: {xgb_best['val_log_loss_mean']:.6f}")
print(f"Models copy feature count: {len(MODELS_COPY_FEATURE_COLS)}")
print(f"XGB params: {json.dumps(XGB_PARAMS, indent=2)}")
print(f"\nRe-eval baseline on current data: {models_copy_baseline['val_log_loss_mean']:.6f}")

Models copy tuned feature label: all_engineered_extended
Models copy tuned val log loss: 0.424062
Models copy feature count: 62
XGB params: {
  "n_estimators": 300,
  "max_depth": 4,
  "learning_rate": 0.021972740562503774,
  "subsample": 0.7758253794788034,
  "colsample_bytree": 0.8970303337810756,
  "reg_lambda": 1.0513964757652297,
  "reg_alpha": 4.319755742829764e-05,
  "min_child_weight": 7
}

Re-eval baseline on current data: 0.424062


## 3. Ablation block definitions

**Base blocks** (from `models copy.ipynb`):
`demographics`, `delay_engineered`, `pay_status`, `pay_amounts`, `bill_amounts`, `credit_util`, `bill_trends`, `models_copy_new_engineered`

**New feature_eng blocks**:
`delay_trends`, `pay_amt_stats`, `payment_change`, `util_stats`, `delay_util_interactions`

In [17]:
print(f"Base ablation blocks ({len(BASE_ABLATION_BLOCKS)}):")
for name, cols in BASE_ABLATION_BLOCKS.items():
    print(f"  {name}: {len(cols)} cols")

print(f"\nNew ablation blocks ({len(NEW_ABLATION_BLOCKS)}):")
for name, cols in NEW_ABLATION_BLOCKS.items():
    print(f"  {name}: {cols}")

full_feature_cols = build_feature_cols_from_blocks(all_blocks_active())
print(f"\nAll blocks ON -> {len(full_feature_cols)} features")

Base ablation blocks (8):
  demographics: 5 cols
  delay_engineered: 5 cols
  pay_status: 6 cols
  pay_amounts: 7 cols
  bill_amounts: 7 cols
  credit_util: 6 cols
  bill_trends: 13 cols
  models_copy_new_engineered: 13 cols

New ablation blocks (5):
  delay_trends: ['recent_delay_mean', 'old_delay_mean', 'delay_deterioration_v2', 'weighted_delay_v2']
  pay_amt_stats: ['mean_pay_amt', 'std_pay_amt', 'max_pay_amt', 'num_zero_payments']
  payment_change: ['recent_pay_mean', 'old_pay_mean', 'payment_change_recent']
  util_stats: ['mean_util', 'max_util', 'std_util', 'months_high_util', 'months_over_limit', 'recent_util_vs_avg']
  delay_util_interactions: ['recent_delay_x_util', 'delay_count_x_util', 'severe_delay_x_util']

All blocks ON -> 82 features


## 4. Fixed-parameter ablation (models copy XGB params)

Uses tuned hyperparameters from section 2; only feature blocks vary.

- **Leave-one-out:** all blocks ON, remove one block at a time
- **Add-one-new:** base blocks only, add one new block at a time

Negative `delta_val_log_loss` = improvement.

In [25]:
leave_one_out_report, add_one_new_report = run_fixed_params_ablation(
    train_df, y, XGB_PARAMS
)

print("Leave-one-out ablation (reference: all blocks ON):")
print(leave_one_out_report.to_string(index=False))

print("\nAdd-one-new-block ablation (reference: base blocks only):")
print(add_one_new_report.to_string(index=False))

Leave-one-out ablation (reference: all blocks ON):
                             scenario              block_changed  val_log_loss_mean  val_roc_auc_mean  delta_val_log_loss  n_features
              leave_out: delay_trends               delay_trends           0.423923          0.788503           -0.000248          78
          leave_out: delay_engineered           delay_engineered           0.423942          0.788593           -0.000228          77
               leave_out: bill_trends                bill_trends           0.423990          0.787774           -0.000180          69
   leave_out: delay_util_interactions    delay_util_interactions           0.423994          0.788226           -0.000177          79
            leave_out: payment_change             payment_change           0.424094          0.788518           -0.000076          79
                        all_blocks_on                                      0.424171          0.788309            0.000000          82
           

## 5. Optuna setup — joint params + blocks

Search space per trial:
- **Hyperparameters:** same ranges as `build_xgb_classifier()` in models copy
- **Feature blocks:** on/off toggle for each block in `ALL_ABLATION_BLOCKS`

Uses a separate SQLite DB so it does not conflict with `models copy` studies.

In [26]:
feature_eng_study = optuna.create_study(
    study_name="xgb_joint_params_and_blocks",
    storage=OPTUNA_STORAGE,
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_RANDOM_STATE),
)

joint_objective = make_joint_optuna_objective(train_df, y)

print(f"Optuna storage: {OPTUNA_STORAGE}")
print(f"Study: {feature_eng_study.study_name}")
print(f"Planned trials: {OPTUNA_N_TRIALS}")
print(f"Block toggles: {len(ALL_ABLATION_BLOCKS)}")
print(f"XGB params tuned: n_estimators, max_depth, learning_rate, subsample,")
print(f"                  colsample_bytree, reg_lambda, reg_alpha, min_child_weight")

Optuna storage: sqlite:///optuna_feature_eng_joint.db
Study: xgb_joint_params_and_blocks
Planned trials: 50
Block toggles: 13
XGB params tuned: n_estimators, max_depth, learning_rate, subsample,
                  colsample_bytree, reg_lambda, reg_alpha, min_child_weight


## 6. Run joint Optuna search (manual)

Run when ready — can take several minutes.

In [29]:
feature_eng_study.optimize(
    joint_objective,
    n_trials=200,
    show_progress_bar=True,
)

best = feature_eng_study.best_trial
print(f"Best val log loss: {best.value:.6f}")
print(f"Best enabled blocks: {best.user_attrs['enabled_blocks']}")
print(f"Best XGB params: {best.user_attrs['xgb_params']}")

Best trial: 231. Best value: 0.423229: 100%|██████████| 200/200 [22:25<00:00,  6.73s/it]

Best val log loss: 0.423229
Best enabled blocks: ['demographics', 'pay_status', 'bill_amounts', 'bill_trends', 'models_copy_new_engineered', 'delay_trends', 'util_stats']
Best XGB params: {'n_estimators': 450, 'max_depth': 5, 'learning_rate': 0.012848810090718452, 'subsample': 0.642226359148273, 'colsample_bytree': 0.6141750857260327, 'reg_lambda': 6.590179660339618, 'reg_alpha': 3.211528472306935e-08, 'min_child_weight': 9}


## 7. Block impact report (from joint Optuna trials)

For each block: mean val log loss when block is **ON** vs **OFF** across all completed trials.

- Negative `delta_on_minus_off` → block helps when enabled
- Compare `delta_vs_reference_when_on` to models copy baseline from section 2

In [30]:
reference_val = models_copy_baseline["val_log_loss_mean"]
block_effects = summarize_block_effects(feature_eng_study, reference_val)

FEATURE_ENG_BEST = store_feature_eng_best(feature_eng_study, reference_val)

comparison = pd.DataFrame(
    [
        {
            "source": "models_copy_tuned_baseline",
            "val_log_loss_mean": reference_val,
            "val_roc_auc_mean": models_copy_baseline["val_roc_auc_mean"],
            "n_features": len(MODELS_COPY_FEATURE_COLS),
        },
        {
            "source": "feature_eng_joint_best",
            "val_log_loss_mean": FEATURE_ENG_BEST["best_val_log_loss_mean"],
            "val_roc_auc_mean": FEATURE_ENG_BEST["best_val_roc_auc_mean"],
            "n_features": FEATURE_ENG_BEST["n_features"],
        },
    ]
)

print(f"Saved {FEATURE_ENG_BEST_PATH}")
print("\nModels copy baseline vs feature_eng joint best:")
print(comparison.to_string(index=False))

print("\nPer-block effects (negative delta_on_minus_off => block helps):")
print(block_effects.to_string(index=False))

block_effects

Saved feature_eng_joint_best.json

Models copy baseline vs feature_eng joint best:
                    source  val_log_loss_mean  val_roc_auc_mean  n_features
models_copy_tuned_baseline           0.424062          0.788286          62
    feature_eng_joint_best           0.423229          0.789595          54

Per-block effects (negative delta_on_minus_off => block helps):
                     block block_group  trials_with_block_on  trials_with_block_off  mean_val_log_loss_block_on  mean_val_log_loss_block_off  delta_on_minus_off  improves_when_on  delta_vs_reference_when_on
models_copy_new_engineered        base                   289                     11                    0.424746                     0.437480           -0.012734              True                    0.000683
                util_stats feature_eng                   289                     11                    0.424761                     0.437067           -0.012306              True                    0.000699
   

,block,block_group,trials_with_block_on,trials_with_block_off,mean_val_log_loss_block_on,mean_val_log_loss_block_off,delta_on_minus_off,improves_when_on,delta_vs_reference_when_on
7,models_copy_new_engineered,base,289,11,0.424746,0.437480,-0.012734,True,0.000683
11,util_stats,feature_eng,289,11,0.424761,0.437067,-0.012306,True,0.000699
2,pay_status,base,288,12,0.424843,0.434082,-0.009239,True,0.000781
0,demographics,base,284,16,0.424796,0.432601,-0.007805,True,0.000734
4,bill_amounts,base,287,13,0.424976,0.430436,-0.005460,True,0.000914
12,delay_util_interactions,feature_eng,218,82,0.424819,0.426258,-0.001439,True,0.000757
8,delay_trends,feature_eng,154,146,0.424669,0.425786,-0.001117,True,0.000607
6,bill_trends,base,230,70,0.425082,0.425642,-0.000560,True,0.001020
5,credit_util,base,126,174,0.425877,0.424731,0.001146,False,0.001815
1,delay_engineered,base,24,276,0.427420,0.425021,0.002400,False,0.003358


## 8. Test submission (best joint combination)

Train tuned XGB on full train data with best feature cols + params; write `submission_feature_eng.csv`.

**Minimal run:** sections 1 → 2 → 4 → 8 (requires `feature_eng_joint_best.json` from section 7, or from a prior run).

In [31]:
# Uses FEATURE_ENG_BEST from section 7 if available, else loads feature_eng_joint_best.json
if "FEATURE_ENG_BEST" not in globals() or not FEATURE_ENG_BEST:
    FEATURE_ENG_BEST = load_feature_eng_best(FEATURE_ENG_BEST_PATH)

test_df = engineer_all_features(load_data("test.csv"))

submission_feature_eng = make_feature_eng_submission(
    train_df,
    test_df,
    y,
    best_config=FEATURE_ENG_BEST,
    output_path="submission_feature_eng.csv",
)

submission_feature_eng.head()

Saved submission_feature_eng.csv (6000 rows, mean prob=0.219190, 54 features)


,client_id,default_probability
0,CC_0012A082B7B7,0.128776
1,CC_0012BC27DFB6,0.125161
2,CC_001563B2143D,0.095551
3,CC_001B46930B6F,0.093849
4,CC_001D771E1C9F,0.042839
